In [2]:
import os
import json
import math
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error


SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


BASE_DIR = Path(".")
ARTIFACTS_DIR = BASE_DIR / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"

DATA_PATH = Path("S12-hw-dataset.csv")

TARGET_COL = "target"
DATE_COL = "date"

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

WINDOW_SIZE = 14
HORIZON = 1
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.0
LR = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5
MOVING_AVG_WINDOW = 7


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dirs():
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0


def evaluate_forecast(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(rmse(y_true, y_pred)),
        "mape": float(mape(y_true, y_pred)),
    }


def load_data(path: Path):
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    return df


def temporal_split(df: pd.DataFrame):
    n = len(df)
    train_end = int(n * TRAIN_RATIO)
    val_end = int(n * (TRAIN_RATIO + VAL_RATIO))
    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()
    return train_df, val_df, test_df


def plot_series_split(df, train_df, val_df, test_df, save_path: Path):
    plt.figure(figsize=(14, 5))
    plt.plot(train_df[DATE_COL], train_df[TARGET_COL], label="train")
    plt.plot(val_df[DATE_COL], val_df[TARGET_COL], label="validation")
    plt.plot(test_df[DATE_COL], test_df[TARGET_COL], label="test")
    plt.title("Series split")
    plt.xlabel("date")
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def add_time_features(df: pd.DataFrame):
    out = df.copy()
    out["lag_1"] = out[TARGET_COL].shift(1)
    out["lag_7"] = out[TARGET_COL].shift(7)
    out["lag_14"] = out[TARGET_COL].shift(14)
    out["rolling_mean_7"] = out[TARGET_COL].shift(1).rolling(7).mean()
    out["rolling_std_7"] = out[TARGET_COL].shift(1).rolling(7).std()
    out["day_of_week"] = out[DATE_COL].dt.dayofweek
    return out


def prepare_feature_df(df: pd.DataFrame):
    feat_df = add_time_features(df)
    feat_df = feat_df.dropna().reset_index(drop=True)
    return feat_df


def split_feature_df(feat_df: pd.DataFrame, train_end_date, val_end_date):
    train_feat = feat_df[feat_df[DATE_COL] <= train_end_date].copy()
    val_feat = feat_df[(feat_df[DATE_COL] > train_end_date) & (feat_df[DATE_COL] <= val_end_date)].copy()
    test_feat = feat_df[feat_df[DATE_COL] > val_end_date].copy()
    return train_feat, val_feat, test_feat


def get_feature_columns(df: pd.DataFrame):
    return ["lag_1", "lag_7", "lag_14", "rolling_mean_7", "rolling_std_7", "day_of_week"]


def run_b1_naive_last(val_df: pd.DataFrame, test_df: pd.DataFrame):
    val_pred = val_df[TARGET_COL].shift(1).dropna().values
    val_true = val_df[TARGET_COL].iloc[1:].values

    test_pred = test_df[TARGET_COL].shift(1).dropna().values
    test_true = test_df[TARGET_COL].iloc[1:].values

    val_metrics = evaluate_forecast(val_true, val_pred)
    test_metrics = evaluate_forecast(test_true, test_pred)

    return {
        "val_true": val_true,
        "val_pred": val_pred,
        "test_true": test_true,
        "test_pred": test_pred,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


def run_b2_moving_average(full_df: pd.DataFrame, train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, window: int = 7):
    df = full_df.copy()
    df["ma_pred"] = df[TARGET_COL].shift(1).rolling(window).mean()

    val_part = df[df[DATE_COL].isin(val_df[DATE_COL])].dropna().copy()
    test_part = df[df[DATE_COL].isin(test_df[DATE_COL])].dropna().copy()

    val_true = val_part[TARGET_COL].values
    val_pred = val_part["ma_pred"].values

    test_true = test_part[TARGET_COL].values
    test_pred = test_part["ma_pred"].values

    val_metrics = evaluate_forecast(val_true, val_pred)
    test_metrics = evaluate_forecast(test_true, test_pred)

    return {
        "val_true": val_true,
        "val_pred": val_pred,
        "test_true": test_true,
        "test_pred": test_pred,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


def run_b3_ridge(full_df: pd.DataFrame, train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    feat_df = prepare_feature_df(full_df)
    train_end_date = train_df[DATE_COL].max()
    val_end_date = val_df[DATE_COL].max()

    train_feat, val_feat, test_feat = split_feature_df(feat_df, train_end_date, val_end_date)

    feature_cols = get_feature_columns(feat_df)

    X_train = train_feat[feature_cols].values
    y_train = train_feat[TARGET_COL].values

    X_val = val_feat[feature_cols].values
    y_val = val_feat[TARGET_COL].values

    X_test = test_feat[feature_cols].values
    y_test = test_feat[TARGET_COL].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    model = Ridge(alpha=1.0)
    model.fit(X_train_scaled, y_train)

    val_pred = model.predict(X_val_scaled)
    test_pred = model.predict(X_test_scaled)

    val_metrics = evaluate_forecast(y_val, val_pred)
    test_metrics = evaluate_forecast(y_test, test_pred)

    return {
        "model": model,
        "scaler": scaler,
        "feature_cols": feature_cols,
        "val_true": y_val,
        "val_pred": val_pred,
        "test_true": y_test,
        "test_pred": test_pred,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


class SequenceDataset(Dataset):
    def __init__(self, values, window_size=14, horizon=1):
        self.values = values
        self.window_size = window_size
        self.horizon = horizon

    def __len__(self):
        return len(self.values) - self.window_size - self.horizon + 1

    def __getitem__(self, idx):
        x = self.values[idx:idx + self.window_size]
        y = self.values[idx + self.window_size + self.horizon - 1]
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(y, dtype=torch.float32)
        return x, y


class GRUForecast(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.head(out).squeeze(-1)
        return out


def create_seq_loaders(train_df, val_df, test_df, window_size, horizon, batch_size):
    train_values = train_df[TARGET_COL].values.reshape(-1, 1)
    val_values = val_df[TARGET_COL].values.reshape(-1, 1)
    test_values = test_df[TARGET_COL].values.reshape(-1, 1)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_values).flatten()
    val_scaled = scaler.transform(val_values).flatten()
    test_scaled = scaler.transform(test_values).flatten()

    train_ds = SequenceDataset(train_scaled, window_size, horizon)
    val_ds = SequenceDataset(val_scaled, window_size, horizon)
    test_ds = SequenceDataset(test_scaled, window_size, horizon)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, scaler


def predict_loader(model, loader, scaler):
    model.eval()
    preds = []
    targets = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            pred = model(x)

            preds.append(pred.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds).reshape(-1, 1)
    targets = np.concatenate(targets).reshape(-1, 1)

    preds_inv = scaler.inverse_transform(preds).flatten()
    targets_inv = scaler.inverse_transform(targets).flatten()

    return targets_inv, preds_inv


def train_gru(train_loader, val_loader, scaler):
    model = GRUForecast(
        input_size=1,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_model_state = None
    best_val_mae = float("inf")
    patience_counter = 0

    history = {
        "train_loss": [],
        "val_mae": [],
        "val_rmse": [],
        "val_mape": [],
    }

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_losses = []

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_true, val_pred = predict_loader(model, val_loader, scaler)
        val_metrics = evaluate_forecast(val_true, val_pred)

        history["train_loss"].append(train_loss)
        history["val_mae"].append(val_metrics["mae"])
        history["val_rmse"].append(val_metrics["rmse"])
        history["val_mape"].append(val_metrics["mape"])

        if val_metrics["mae"] < best_val_mae:
            best_val_mae = val_metrics["mae"]
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_model_state)
    return model, history


def run_r1_gru(train_df, val_df, test_df):
    train_loader, val_loader, test_loader, scaler = create_seq_loaders(
        train_df, val_df, test_df, WINDOW_SIZE, HORIZON, BATCH_SIZE
    )

    model, history = train_gru(train_loader, val_loader, scaler)

    val_true, val_pred = predict_loader(model, val_loader, scaler)
    test_true, test_pred = predict_loader(model, test_loader, scaler)

    val_metrics = evaluate_forecast(val_true, val_pred)
    test_metrics = evaluate_forecast(test_true, test_pred)

    return {
        "model": model,
        "scaler": scaler,
        "history": history,
        "val_true": val_true,
        "val_pred": val_pred,
        "test_true": test_true,
        "test_pred": test_pred,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


def plot_baselines_compare(results_dict, save_path: Path):
    names = list(results_dict.keys())
    maes = [results_dict[k]["val_metrics"]["mae"] for k in names]

    plt.figure(figsize=(8, 5))
    plt.bar(names, maes)
    plt.title("Validation MAE comparison")
    plt.xlabel("experiment")
    plt.ylabel("MAE")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_learning_curves(history, save_path: Path):
    epochs = np.arange(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_mae"], label="val_mae")
    plt.title("GRU learning curves")
    plt.xlabel("epoch")
    plt.ylabel("value")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_best_forecast_test(y_true, y_pred, save_path: Path):
    plt.figure(figsize=(14, 5))
    plt.plot(y_true, label="fact")
    plt.plot(y_pred, label="forecast")
    plt.title("Best forecast on test")
    plt.xlabel("time_step")
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def save_best_gru(model, save_path: Path):
    torch.save(model.state_dict(), save_path)


def save_best_gru_config(save_path: Path):
    config = {
        "window_size": WINDOW_SIZE,
        "horizon": HORIZON,
        "batch_size": BATCH_SIZE,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "lr": LR,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "seed": SEED,
        "device": str(DEVICE),
    }
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)


def make_runs_dataframe(results, dataset_name):
    rows = []

    for exp_id, res in results.items():
        row = {
            "experiment_id": exp_id,
            "task": "forecasting",
            "dataset": dataset_name,
            "seed": SEED,
            "split_summary": f"{TRAIN_RATIO:.2f}/{VAL_RATIO:.2f}/{TEST_RATIO:.2f}",
            "window_size": WINDOW_SIZE if exp_id == "R1" else np.nan,
            "horizon": HORIZON,
            "model_summary": {
                "B1": "naive-last",
                "B2": f"moving-average-{MOVING_AVG_WINDOW}",
                "B3": "Ridge",
                "R1": f"GRU(hidden={HIDDEN_SIZE}, layers={NUM_LAYERS})",
            }[exp_id],
            "features_summary": {
                "B1": "last_value",
                "B2": f"rolling_mean_{MOVING_AVG_WINDOW}",
                "B3": "lag_1, lag_7, lag_14, rolling_mean_7, rolling_std_7, day_of_week",
                "R1": f"sequence_window_{WINDOW_SIZE}",
            }[exp_id],
            "scaler": {
                "B1": "",
                "B2": "",
                "B3": "StandardScaler(X_train only)",
                "R1": "StandardScaler(train target only)",
            }[exp_id],
            "optimizer": "Adam" if exp_id == "R1" else "",
            "lr": LR if exp_id == "R1" else np.nan,
            "epochs_trained": len(res["history"]["train_loss"]) if exp_id == "R1" else 0,
            "best_val_mae": res["val_metrics"]["mae"],
            "best_val_rmse": res["val_metrics"]["rmse"],
            "best_val_mape": res["val_metrics"]["mape"],
            "test_mae": res["test_metrics"]["mae"],
            "test_rmse": res["test_metrics"]["rmse"],
            "test_mape": res["test_metrics"]["mape"],
            "notes": "",
        }
        rows.append(row)

    return pd.DataFrame(rows)


def main():
    set_seed(SEED)
    ensure_dirs()

    df = load_data(DATA_PATH)
    train_df, val_df, test_df = temporal_split(df)

    plot_series_split(
        df,
        train_df,
        val_df,
        test_df,
        FIGURES_DIR / "series_split.png"
    )

    b1 = run_b1_naive_last(val_df, test_df)
    b2 = run_b2_moving_average(df, train_df, val_df, test_df, MOVING_AVG_WINDOW)
    b3 = run_b3_ridge(df, train_df, val_df, test_df)
    r1 = run_r1_gru(train_df, val_df, test_df)

    results = {
        "B1": b1,
        "B2": b2,
        "B3": b3,
        "R1": r1,
    }

    plot_baselines_compare(results, FIGURES_DIR / "baselines_compare.png")
    plot_learning_curves(r1["history"], FIGURES_DIR / "gru_learning_curves.png")

    best_exp = min(results.items(), key=lambda x: x[1]["val_metrics"]["mae"])[0]
    plot_best_forecast_test(
        results[best_exp]["test_true"],
        results[best_exp]["test_pred"],
        FIGURES_DIR / "best_forecast_test.png"
    )

    save_best_gru(r1["model"], ARTIFACTS_DIR / "best_gru.pt")
    save_best_gru_config(ARTIFACTS_DIR / "best_gru_config.json")

    runs_df = make_runs_dataframe(results, DATA_PATH.name)
    runs_df.to_csv(ARTIFACTS_DIR / "runs.csv", index=False, encoding="utf-8")

    print("Готово")
    print(f"Лучший эксперимент по val_mae: {best_exp}")
    print(runs_df[[
        "experiment_id",
        "best_val_mae",
        "best_val_rmse",
        "best_val_mape",
        "test_mae",
        "test_rmse",
        "test_mape"
    ]])


if __name__ == "__main__":
    main()

Готово
Лучший эксперимент по val_mae: R1
  experiment_id  best_val_mae  best_val_rmse  best_val_mape   test_mae  \
0            B1      6.438501       8.196911       4.391461   6.340958   
1            B2     12.702013      15.217645       8.816864  12.740309   
2            B3      7.188730       8.732282       4.796689   7.337462   
3            R1      6.182077       7.794437       4.151219   6.552131   

   test_rmse  test_mape  
0   8.060187   4.146299  
1  15.238699   8.549006  
2   9.045724   4.699417  
3   8.349936   4.231873  
